In [3]:
# 获取token
import requests
import os

env = "admin"

login_url = f"https://{env}.summerfarm.net/authentication/auth/username/login"
login_data = {
    "username": "peng.tang@summerfarm.net",
    "password": os.getenv(
        f"XIANMU_ADMIN_PASSWORD_{env}", os.getenv("XIANMU_ADMIN_PASSWORD")
    ),
}

token = requests.post(login_url, data=login_data).json()

print(token)

token_str = token.get("data").get("token")

headers = {
    "token": token_str,
    "xm-rqid": "批量修改春节运费",
    "xm-uid": "2047",
    "Content-Type": "application/json;charset=UTF-8",
}

print(headers)

{'code': 'SUCCESS', 'data': {'authId': 2082, 'bizUserId': 2047, 'phone': '18618107293', 'realname': '唐鹏', 'token': 'admin__469ebd09-af7e-4fab-89b9-333bc7fc9490', 'userBaseId': 2080}, 'msg': '', 'success': True}
{'token': 'admin__469ebd09-af7e-4fab-89b9-333bc7fc9490', 'xm-rqid': '批量修改春节运费', 'xm-uid': '2047', 'Content-Type': 'application/json;charset=UTF-8'}


In [4]:
import requests
import json

area_no_list_to_update = [
    44262,
    2750,
    44225,
    1818,
    14816,
    19610,
    36137,
    1151,
    19415,
    25367,
    8246,
    14400,
    44229,
    44219,
    44220,
    44218,
    17006,
    44158,
    44123,
    44171,
    18637,
    20882,
    40766,
    25541,
    31164,
    44230,
    10358,
    44127,
    44191,
    9066,
    13351,
    44161,
    8254,
    18268,
    44207,
    18367,
    44157,
    44250,
    9606,
    9773,
    44128,
    14343,
    44210,
    9137,
    44260,
    1002,
    44129,
    37403,
    8468,
    24635,
    33858,
    9167,
    9400,
    44163,
    44209,
    1001,
    44235,
    44178,
    44177,
    10111,
    44125,
    44264,
    44180,
    43704,
    9376,
    39867,
    33611,
    44237,
    44254,
    10287,
    14564,
    44145,
    41224,
    25624,
    44212,
    44213,
    18704,
    44216,
    44116,
    25105,
    44234,
    38077,
    44236,
    1003,
    44148,
    44217,
    44144,
    44221,
    44154,
    9585,
    32751,
    20210,
    33016,
    34270,
    44136,
    21060,
    20504,
    10416,
    34663,
    17634,
    23575,
    44138,
    1881,
    17574,
    44238,
    1666,
    44134,
    17367,
    20627,
    44135,
    24548,
    14740,
    20047,
    44119,
    2048,
    20841,
    8577,
    17260,
    19462,
    20264,
    44137,
    44133,
    1218,
    44176,
    44259,
    29765,
    44146,
    35714,
    2767,
    32484,
    18872,
    44122,
    44179,
    44253,
    44130,
    44147,
]

detail_url = (
    "https://admin.summerfarm.net/marketing-center/delivery-fee-rule/query/detail"
)


def get_area_delivery_fee_detail(
    area_no: int = 1001, from_local_cache: bool = False
) -> dict:
    if from_local_cache:
        with open("./delivery_fee_rules_backup_20250124.json", "r") as f:
            backup_json = json.load(f)
        return backup_json.get(str(area_no))
    data = {"type": 3, "businessId": area_no}
    headers = {
        "accept": "application/json, text/plain, */*",
        "content-type": "application/json;charset=UTF-8",
        "token": token_str,
        "xm-rqid": "chunjie_delivery_fee_10yuan",
    }
    rules = None
    try:
        response = requests.post(
            "https://admin.summerfarm.net/marketing-center/delivery-fee-rule/query/detail",
            headers=headers,
            json=data,
        )
        response.raise_for_status()  # Raise an HTTPError for bad responses
        rules = response.text
        print(rules)
        rules_json = json.loads(rules)
        if rules_json and "data" in rules_json:
            return rules_json.get("data")
        else:
            return {}
    except requests.exceptions.RequestException as e:
        print(f"HTTP request error: {e}, rules: {rules}")
        return {}
    except UnicodeEncodeError as e:
        print(f"UnicodeEncodeError encountered: {e}, rules: {rules}")
        return {}
    except Exception as e:
        print(f"An error occurred: {e}")
        return {}


objects_used_to_modify = []
no_need_to_modify = []
backup_rules = {}  # Dictionary to store original rules for backup

import copy

for area_no in area_no_list_to_update:
    modified = False
    rules_of_area = get_area_delivery_fee_detail(area_no, from_local_cache=True)
    if not rules_of_area:
        print(f"获取配送规则错误: {area_no}")
        continue

    # Store original rules for backup
    backup_rules[area_no] = copy.deepcopy(rules_of_area)

    # Extract and transform the rules into the desired format
    transformed_rules = {
        "type": rules_of_area.get("type"),
        "businessId": int(rules_of_area.get("businessId")),
        "ruleInputList": [],
    }

    for rule in rules_of_area.get("ruleVOList", []):
        rule_input = {
            "ageing": rule.get("ageing"),
            "startDeliveryAmount": rule.get("startDeliveryAmount"),
            "categoryRuleInputList": [],
        }

        for category_rule in rule.get("categoryRuleVOList", []):
            if (
                category_rule.get("deliveryFee", 0) <= 0
                or category_rule.get("expressFee", 0) <= 0
            ):
                modified = True
            category_rule_input = {
                "stepValue": float(category_rule.get("stepValue")),
                "deliveryFee": (
                    category_rule.get("deliveryFee")
                    if category_rule.get("deliveryFee") > 0.0
                    else 0.0
                ),
                "expressFee": (
                    category_rule.get("expressFee")
                    if category_rule.get("expressFee") > 0.0
                    else 0.0
                ),
                "feeMode": category_rule.get("feeMode"),
                "categoryType": category_rule.get("categoryType"),
            }
            rule_input["categoryRuleInputList"].append(category_rule_input)

        transformed_rules["ruleInputList"].append(rule_input)

    if modified:
        objects_used_to_modify.append(transformed_rules)
    else:
        print(f"这个区域不需要改规则:{area_no}, 规则列表:{rules_of_area}")
        no_need_to_modify.append(rules_of_area)

# Save backup data to a file
from datetime import datetime
yearmonth = datetime.now().strftime("%Y%m")
file_name = f"delivery_fee_rules_backup_{yearmonth}-{env}.json"
with open(f"./{file_name}", "w", encoding="utf-8") as f:
    json.dump(backup_rules, f, ensure_ascii=False, indent=2)

print(f"原始规则已备份到 {file_name}")
print(objects_used_to_modify)

这个区域不需要改规则:44250, 规则列表:{'businessId': '44250', 'ruleVOList': [{'ageing': 0, 'categoryRuleVOList': [{'categoryType': 1, 'deliveryFee': 10.0, 'expressFee': 10.0, 'feeMode': 1, 'stepValue': '0'}], 'startDeliveryAmount': 0.0}, {'ageing': 1, 'categoryRuleVOList': [{'categoryType': 1, 'deliveryFee': 10.0, 'expressFee': 10.0, 'feeMode': 1, 'stepValue': '0'}], 'startDeliveryAmount': 0.0}], 'type': 3}
这个区域不需要改规则:44260, 规则列表:{'businessId': '44260', 'ruleVOList': [{'ageing': 0, 'categoryRuleVOList': [{'categoryType': 1, 'deliveryFee': 10.0, 'expressFee': 10.0, 'feeMode': 1, 'stepValue': '0'}], 'startDeliveryAmount': 0.0}, {'ageing': 1, 'categoryRuleVOList': [{'categoryType': 1, 'deliveryFee': 10.0, 'expressFee': 10.0, 'feeMode': 1, 'stepValue': '0'}], 'startDeliveryAmount': 0.0}], 'type': 3}
这个区域不需要改规则:44235, 规则列表:{'businessId': '44235', 'ruleVOList': [{'ageing': 0, 'categoryRuleVOList': [{'categoryType': 1, 'deliveryFee': 10.0, 'expressFee': 10.0, 'feeMode': 1, 'stepValue': '0'}], 'startDelivery

In [5]:
# import requests

# if len(objects_used_to_modify) > 0:
#     for obj in objects_used_to_modify:
#         url = f"https://{env}.summerfarm.net/marketing-center/delivery-fee-rule/upsert/save"
#         headers = {
#             "accept": "application/json, text/plain, */*",
#             "content-type": "application/json;charset=UTF-8",
#             "token": token_str,
#             "xm-phone": "18618107293",
#             "xm-rqid": "chunjie_update_delivery_fee_10",
#         }

#         response = requests.post(url, headers=headers, json=obj)
#         print(f"Response for obj: {obj}\n{response.status_code}, {response.text}")

In [6]:
for obj in no_need_to_modify:
    print("无须改动:", obj)

无须改动: {'businessId': '44250', 'ruleVOList': [{'ageing': 0, 'categoryRuleVOList': [{'categoryType': 1, 'deliveryFee': 10.0, 'expressFee': 10.0, 'feeMode': 1, 'stepValue': '0'}], 'startDeliveryAmount': 0.0}, {'ageing': 1, 'categoryRuleVOList': [{'categoryType': 1, 'deliveryFee': 10.0, 'expressFee': 10.0, 'feeMode': 1, 'stepValue': '0'}], 'startDeliveryAmount': 0.0}], 'type': 3}
无须改动: {'businessId': '44260', 'ruleVOList': [{'ageing': 0, 'categoryRuleVOList': [{'categoryType': 1, 'deliveryFee': 10.0, 'expressFee': 10.0, 'feeMode': 1, 'stepValue': '0'}], 'startDeliveryAmount': 0.0}, {'ageing': 1, 'categoryRuleVOList': [{'categoryType': 1, 'deliveryFee': 10.0, 'expressFee': 10.0, 'feeMode': 1, 'stepValue': '0'}], 'startDeliveryAmount': 0.0}], 'type': 3}
无须改动: {'businessId': '44235', 'ruleVOList': [{'ageing': 0, 'categoryRuleVOList': [{'categoryType': 1, 'deliveryFee': 10.0, 'expressFee': 10.0, 'feeMode': 1, 'stepValue': '0'}], 'startDeliveryAmount': 0.0}, {'ageing': 1, 'categoryRuleVOList': 